In [1]:
import os
os.chdir("../")
%pwd

'd:\\Deep_Learning_Object_Detection\\MLOPs\\pneumonia-segmentation'

In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir:    Path
    source_type: str            
    source:      str     
    name:        str

In [3]:
from pneumonia_segmentation.constants import *
from pneumonia_segmentation.utils.common import read_yaml, create_directories

from dotenv import load_dotenv
load_dotenv()

class ConfigurationManger:
    def __init__(self, 
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion_config
        create_directories([config.root_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir    = config.root_dir,
            source_type = os.getenv("DATA_SOURCE_TYPE", "LOCAL"),
            source      = os.getenv("DATA_SOURCE"),
            name        = os.getenv("DATA_SOURCE_NAME"),
        )

        return data_ingestion_config

In [4]:
import os, sys
from pneumonia_segmentation import logging
from pneumonia_segmentation.exception import CustomException
from pneumonia_segmentation.adapters import BaseDataIngestionAdapter
from pneumonia_segmentation.adapters.local_ingestion_adapter import LocalIngestionAdapter
from pneumonia_segmentation.adapters.kaggle_ingestion_adapter import KaggleIngestionAdapter

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config  = config
        self.adapter = self._get_ingestion_adapter()
    
    def _get_ingestion_adapter(self) -> BaseDataIngestionAdapter:
        source_type = self.config.source_type
        source      = self.config.source
        
        if source_type == "LOCAL":
            return LocalIngestionAdapter(source)
        elif source_type == "KAGGLE":
            return KaggleIngestionAdapter(source)
        else:
            raise ValueError(f"Source type {source_type} not supported")
        
    def fetch_data(self) -> None:
        try: 
            dst = os.path.join(self.config.root_dir, self.config.name)
            self.adapter.fetch(dst)
            logging.info(f"Data fetched at {dst}")
        except Exception as e:
            raise CustomException(e, sys)

In [5]:
try:
    config = ConfigurationManger()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.fetch_data()
except Exception as e:
    raise CustomException(e, sys)

[2026-04-17 12:06:07,914: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-17 12:06:07,918: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-17 12:06:07,920: INFO: common: created directory at: artifacts]
[2026-04-17 12:06:07,937: INFO: common: created directory at: artifacts/data_ingestion]
[2026-04-17 12:06:07,947: INFO: kaggle_ingestion_adapter: Downloading Kaggle dataset andrewmvd/covid19-ct-scans -> artifacts/data_ingestion\COVID-19_CT_SCANS]
[2026-04-17 12:07:24,741: INFO: 3156432644: Data fetched at artifacts/data_ingestion\COVID-19_CT_SCANS]
